# Fine-Tune Qwen 7B with Unsloth (Free Colab)

Open-source fine-tuning using:
- **Unsloth** (2-5x faster training)
- **QLoRA** (4-bit quantization)
- **Qwen 2.5 7B** (strong multilingual model)

Works on Google Colab free tier (T4 GPU, 16 GB VRAM).

## Steps
1. Install dependencies
2. Upload your training data
3. Fine-tune the model
4. Export for Ollama (run on your MacBook)

In [ ]:
# Step 1: Install Unsloth and dependencies
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# Step 2: Upload your training data
# Option A: Upload manually via Colab file browser (left sidebar)
# Option B: Upload programmatically
from google.colab import files
import os

# Upload your training_data.jsonl file
if not os.path.exists("training_data.jsonl"):
    print("Upload your training_data.jsonl file:")
    uploaded = files.upload()
    print(f"Uploaded: {list(uploaded.keys())}")
else:
    print("training_data.jsonl already exists")

In [ ]:
# Step 3: Load the base model with Unsloth
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect
    load_in_4bit=True,  # QLoRA 4-bit quantization
)

print(f"Model loaded: {MODEL_NAME}")
print(f"GPU memory used: {round(model.get_memory_footprint() / 1e9, 2)} GB")

In [ ]:
# Step 4: Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank - higher = more capacity, more memory
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,  # Unsloth optimized - use 0
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less memory
    random_state=42,
)

# Show trainable parameters
model.print_trainable_parameters()

In [ ]:
# Step 5: Load and format training data
import json
from datasets import Dataset

def load_training_data(path="training_data.jsonl"):
    """Load JSONL training data."""
    data = []
    with open(path) as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def format_chat(example):
    """Format conversations into the chat template."""
    text = tokenizer.apply_chat_template(
        example["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

# Load data
raw_data = load_training_data("training_data.jsonl")
dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_chat)

print(f"Training examples: {len(dataset)}")
print(f"\nSample formatted text:\n{dataset[0]['text'][:500]}")

In [ ]:
# Step 6: Configure training
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,  # Set True if you have many short examples
    args=TrainingArguments(
        # Training hyperparameters
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",

        # Memory optimization
        fp16=True,
        bf16=False,  # T4 doesn't support bf16
        optim="adamw_8bit",

        # Logging
        logging_steps=1,
        output_dir="outputs",
        save_strategy="epoch",
        seed=42,
    ),
)

print("Trainer configured. Ready to fine-tune!")

In [ ]:
# Step 7: Train!
print("Starting fine-tuning...")
stats = trainer.train()

print(f"\nTraining complete!")
print(f"Total steps: {stats.global_step}")
print(f"Training loss: {stats.training_loss:.4f}")
print(f"Runtime: {stats.metrics['train_runtime']:.0f} seconds")

In [ ]:
# Step 8: Test the fine-tuned model
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "You are a helpful product expert."},
    {"role": "user", "content": "How much does your product cost?"},
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(f"Q: How much does your product cost?")
print(f"A: {response}")

In [ ]:
# Step 9: Export to GGUF for Ollama (run on your MacBook)
# This converts the model to a format your M2 MacBook Air can run

EXPORT_NAME = "my-product-expert"

# Save as GGUF Q4_K_M (good quality, ~4 GB file size)
model.save_pretrained_gguf(
    EXPORT_NAME,
    tokenizer,
    quantization_method="q4_k_m",
)

print(f"\nModel exported to: {EXPORT_NAME}/")
print("Download this folder and use with Ollama on your MacBook.")

In [ ]:
# Step 10: Download the model file
import glob

gguf_files = glob.glob(f"{EXPORT_NAME}/*.gguf")
if gguf_files:
    print(f"Downloading: {gguf_files[0]}")
    files.download(gguf_files[0])
else:
    print("GGUF file not found. Check the export step above.")

# Next Steps: Run on Your MacBook

1. Install Ollama: `brew install ollama`
2. Create a Modelfile (see `setup_ollama.sh` in the repo)
3. Import: `ollama create my-product-expert -f Modelfile`
4. Run: `ollama run my-product-expert`

The Q4_K_M model is ~4 GB and runs comfortably on an M2 MacBook Air with 8 GB RAM.